# LeetCode 1514 — Path with Maximum Probability bằng Python

Link đề: https://leetcode.com/problems/path-with-maximum-probability/ · Độ khó: **Medium** · Chủ đề: Array, Graph Theory, Heap (Priority Queue), Shortest Path, Dijkstra's Algorithm

Skeleton LeetCode:

```python
class Solution:
    def maxProbability(self, n: int, edges: list[list[int]], succProb: list[float], start_node: int, end_node: int) -> float:
        ...
```

## 1. Mục tiêu học

1. Biểu diễn **đồ thị vô hướng có trọng số** bằng danh sách kề (`list` các `list` cặp `(hàng xóm, xác suất)`).
2. Hiểu vì sao bài "xác suất lớn nhất" là bài **đường đi ngắn nhất trá hình**: nhân xác suất ∈ [0, 1] chỉ làm giảm giá trị, giống cộng trọng số không âm chỉ làm tăng.
3. Viết được 4 cách: DFS mọi đường đi đơn (brute force), **Bellman–Ford**, **Dijkstra với max-heap** (đẩy `-prob`), Dijkstra trên `-log(prob)`.
4. Biết cách so sánh **số thực** với sai số (`abs(a − b) ≤ 1e-5`), không dùng `==`.
5. Tránh các lỗi kinh điển: quên cạnh hai chiều, dùng min-heap theo xác suất, khởi tạo sai xác suất của đỉnh xuất phát.

## 2. Đề bài & ví dụ

Cho đồ thị **vô hướng** `n` đỉnh (đánh số từ 0). `edges[i] = [a, b]` là cạnh nối `a`–`b` với xác suất đi qua thành công `succProb[i]`. Cho hai đỉnh `start` và `end` (tham số `start_node`, `end_node`), tìm đường đi có **xác suất thành công lớn nhất** (tích các xác suất trên đường) và trả về xác suất đó. Nếu không có đường, trả `0`. Đáp án được chấp nhận nếu sai khác không quá `1e-5`.

**Ví dụ 1:** `n = 3, edges = [[0,1],[1,2],[0,2]], succProb = [0.5,0.5,0.2], start = 0, end = 2` → `0.25` (đường 0→1→2 có `0.5·0.5 = 0.25` > `0.2`).

**Ví dụ 2:** như trên nhưng `succProb = [0.5,0.5,0.3]` → `0.3`.

**Ví dụ 3:** `n = 3, edges = [[0,1]], succProb = [0.5], start = 0, end = 2` → `0` (không có đường).

**Ràng buộc:** `2 ≤ n ≤ 10⁴`, `start ≠ end`, `0 ≤ len(edges) = len(succProb) ≤ 2·10⁴`, `0 ≤ succProb[i] ≤ 1`, giữa hai đỉnh có tối đa một cạnh.

In [1]:
import heapq
import math
import random
import time
from collections import deque
from typing import List

EPS = 1e-5
examples = [
    ((3, [[0, 1], [1, 2], [0, 2]], [0.5, 0.5, 0.2], 0, 2), 0.25),
    ((3, [[0, 1], [1, 2], [0, 2]], [0.5, 0.5, 0.3], 0, 2), 0.30),
    ((3, [[0, 1]], [0.5], 0, 2), 0.0),
]
def close(a, b, eps=EPS):
    return abs(a - b) <= eps

for args, want in examples:
    print(args, "->", want)

(3, [[0, 1], [1, 2], [0, 2]], [0.5, 0.5, 0.2], 0, 2) -> 0.25
(3, [[0, 1], [1, 2], [0, 2]], [0.5, 0.5, 0.3], 0, 2) -> 0.3
(3, [[0, 1]], [0.5], 0, 2) -> 0.0


## 3. Kiến thức nền

### 3.1 Danh sách kề cho đồ thị vô hướng có trọng số

Mỗi cạnh `a–b` với xác suất `p` được thêm **hai lần**: `g[a].append((b, p))` và `g[b].append((a, p))`.

In [2]:
def build_graph(n, edges, probs):
    g = [[] for _ in range(n)]
    for (a, b), p in zip(edges, probs):
        g[a].append((b, p))
        g[b].append((a, p))
    return g

g = build_graph(*examples[0][0][:3])
for u, nbrs in enumerate(g):
    print(u, "->", nbrs)

0 -> [(1, 0.5), (2, 0.2)]
1 -> [(0, 0.5), (2, 0.5)]
2 -> [(1, 0.5), (0, 0.2)]


### 3.2 `heapq` là **min-heap** — muốn max-heap thì đổi dấu

Python chỉ có min-heap. Để luôn lấy xác suất **lớn nhất**, đẩy `(-p, đỉnh)`: `-p` nhỏ nhất ⇔ `p` lớn nhất.

In [3]:
h = []
for p, v in [(0.25, 2), (0.5, 1), (0.2, 3), (1.0, 0)]:
    heapq.heappush(h, (-p, v))
while h:
    negp, v = heapq.heappop(h)
    print(f"lấy đỉnh {v} với p = {-negp}")

lấy đỉnh 0 với p = 1.0
lấy đỉnh 1 với p = 0.5
lấy đỉnh 2 với p = 0.25
lấy đỉnh 3 với p = 0.2


### 3.3 So sánh số thực

Phép tính số thực có sai số làm tròn; `0.1 + 0.2 == 0.3` là `False`. Dùng `abs(a − b) ≤ eps` hoặc `math.isclose`.

In [4]:
print(0.1 + 0.2 == 0.3, 0.1 + 0.2)
print(close(0.1 + 0.2, 0.3), math.isclose(0.1 + 0.2, 0.3))
print(0.5 * 0.5, 0.7 * 0.7 * 0.7)

False 0.30000000000000004
True True
0.25 0.3429999999999999


### 3.4 Logarit biến tích thành tổng

`log(p₁·p₂·…) = log p₁ + log p₂ + …`. Vì `0 < p ≤ 1` ⇒ `−log p ≥ 0`. Tối đa hoá tích ⇔ tối thiểu hoá tổng `−log p` (trọng số không âm) → đúng bài Dijkstra chuẩn. Cạnh `p = 0` thì bỏ qua (không bao giờ có lợi).

In [5]:
ps = [0.5, 0.5]
print("tích:", math.prod(ps), "| exp(-tổng(-log)):", math.exp(-sum(-math.log(p) for p in ps)))
print("-log(1) =", -math.log(1.0), "| -log(0.2) =", round(-math.log(0.2), 4))

tích: 0.25 | exp(-tổng(-log)): 0.25
-log(1) = -0.0 | -log(0.2) = 1.6094


### 3.5 Duyệt đồ thị cơ bản (BFS) để kiểm tra liên thông

Nếu `end` không đến được từ `start` thì đáp án 0 — BFS là công cụ kiểm tra nhanh.

In [6]:
def reachable(g, s, t):
    seen = {s}
    q = deque([s])
    while q:
        u = q.popleft()
        if u == t:
            return True
        for v, _ in g[u]:
            if v not in seen:
                seen.add(v)
                q.append(v)
    return False

g3 = build_graph(*examples[2][0][:3])
print(reachable(g3, 0, 1), reachable(g3, 0, 2))

True False


## 4. Ý tưởng nền tảng / trực giác

Gọi `best[v]` = xác suất lớn nhất để đi từ `start` đến `v`. Ta có `best[start] = 1`.

**Quan sát:** đi thêm một cạnh nhân thêm `p ≤ 1` → xác suất **không tăng**. Đây đúng là tính chất "đi thêm cạnh không làm đường ngắn hơn" của trọng số không âm → thuật toán tham lam của **Dijkstra** áp dụng được: đỉnh có `best` lớn nhất chưa chốt có thể chốt ngay.

Tính tay Ví dụ 1 (`0–1: 0.5`, `1–2: 0.5`, `0–2: 0.2`):

| bước | chốt đỉnh (best) | cập nhật hàng xóm | best sau bước |
|---|---|---|---|
| 0 | — | — | [1, 0, 0] |
| 1 | 0 (1.0) | 1 ← 0.5, 2 ← 0.2 | [1, 0.5, 0.2] |
| 2 | 1 (0.5) | 2 ← max(0.2, 0.5·0.5 = 0.25) | [1, 0.5, 0.25] |
| 3 | 2 (0.25) | — | [1, 0.5, 0.25] |

Đáp án `best[2] = 0.25`.

In [7]:
best = [1.0, 0.0, 0.0]
best[1] = max(best[1], best[0] * 0.5); best[2] = max(best[2], best[0] * 0.2)
print("sau khi chốt 0:", best)
best[2] = max(best[2], best[1] * 0.5)
print("sau khi chốt 1:", best)

sau khi chốt 0: [1.0, 0.5, 0.2]
sau khi chốt 1: [1.0, 0.5, 0.25]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** DFS liệt kê **mọi đường đi đơn** (không lặp đỉnh) từ `start` đến `end`, lấy tích lớn nhất. Lặp đỉnh không bao giờ có lợi (nhân thêm số ≤ 1), nên chỉ cần đường đơn.

In [8]:
def brute(n, edges, succProb, start_node, end_node):
    g = build_graph(n, edges, succProb)
    best = 0.0
    visited = [False] * n

    def dfs(u, prob):
        nonlocal best
        if u == end_node:
            best = max(best, prob)
            return
        visited[u] = True
        for v, p in g[u]:
            if not visited[v]:
                dfs(v, prob * p)
        visited[u] = False

    dfs(start_node, 1.0)
    return best

for args, want in examples:
    print(round(brute(*args), 5), "kỳ vọng", want)

0.25 kỳ vọng 0.25
0.3 kỳ vọng 0.3
0.0 kỳ vọng 0.0


**Độ phức tạp:** số đường đi đơn có thể lên tới `O(n!)` (đồ thị đầy đủ) → Time mũ, Space `O(n)` (độ sâu đệ quy). Chỉ dùng cho đồ thị ≤ 8 đỉnh để đối chiếu.

**Nút thắt:** cùng một đỉnh trung gian bị khám phá lại vô số lần qua các đường khác nhau. Chỉ cần nhớ **xác suất tốt nhất** tới mỗi đỉnh (quy hoạch động / nới lỏng cạnh).

## 6. Ý tưởng thuật toán / bước đột phá

**Phép nới lỏng (relax):** với cạnh `u–v` xác suất `p`: *nếu* `best[u]·p > best[v]` *thì* `best[v] = best[u]·p`.

- **Bellman–Ford:** nới lỏng **mọi cạnh** (cả hai chiều) lặp lại tối đa `n − 1` vòng; dừng sớm khi một vòng không thay đổi gì. Đúng vì đường tối ưu là đường đơn ≤ `n − 1` cạnh.
- **Dijkstra (max-heap):** luôn lấy đỉnh có `best` **lớn nhất** chưa chốt. **Bất biến:** khi một đỉnh được lấy ra lần đầu (với giá trị khớp `best`), giá trị đó là tối ưu — vì mọi đường khác phải đi qua đỉnh chưa chốt có `best` ≤ hiện tại, rồi nhân thêm số ≤ 1.
- **Dijkstra trên `−log p`:** biến đổi chính xác về bài đường đi ngắn nhất với trọng số không âm.

Điều kiện then chốt: mọi `p ∈ [0, 1]`. *Nếu* có `p > 1` *thì* tham lam Dijkstra sai (giống trọng số âm).

## 7. Các cách tiếp theo

### Cách 2 — Bellman–Ford `O(n · E)` (thường dừng sớm)

1. `best = [0]*n`, `best[start] = 1`.
2. Lặp tối đa `n − 1` vòng: với mỗi cạnh `(a, b, p)`, nới lỏng cả `a → b` và `b → a`.
3. Vòng nào không đổi thì dừng.

In [9]:
def bellman_ford(n, edges, succProb, start_node, end_node):
    best = [0.0] * n
    best[start_node] = 1.0
    for _ in range(n - 1):
        changed = False
        for (a, b), p in zip(edges, succProb):
            if best[a] * p > best[b]:
                best[b] = best[a] * p
                changed = True
            if best[b] * p > best[a]:
                best[a] = best[b] * p
                changed = True
        if not changed:
            break
    return best[end_node]

for args, want in examples:
    print(round(bellman_ford(*args), 5), "kỳ vọng", want)

0.25 kỳ vọng 0.25
0.3 kỳ vọng 0.3
0.0 kỳ vọng 0.0


In [10]:
class SolutionBF:
    def maxProbability(self, n: int, edges: list[list[int]], succProb: list[float], start_node: int, end_node: int) -> float:
        best = [0.0] * n
        best[start_node] = 1.0
        for _ in range(n - 1):
            changed = False
            for (a, b), p in zip(edges, succProb):
                if best[a] * p > best[b]:
                    best[b] = best[a] * p
                    changed = True
                if best[b] * p > best[a]:
                    best[a] = best[b] * p
                    changed = True
            if not changed:
                break
        return best[end_node]

print([round(SolutionBF().maxProbability(*a), 5) for a, _ in examples])

[0.25, 0.3, 0.0]


**Độ phức tạp:** Time worst `O(n · E)` (`10⁴ · 2·10⁴ = 2·10⁸` — có thể quá chậm trong Python ở trường hợp xấu), nhưng nhờ dừng sớm thường chỉ cần vài chục vòng. Space `O(n)`.

### Cách 3 — Dijkstra với max-heap (đẩy `-prob`) `O(E log V)`

1. Dựng danh sách kề.
2. `best[start] = 1`, heap `[(-1.0, start)]`.
3. Pop `(-p, u)`. Nếu `p < best[u]` thì bỏ qua (bản ghi cũ). Nếu `u == end` trả `p` ngay (đã tối ưu).
4. Với mỗi `(v, w)` kề `u`: nếu `p·w > best[v]` thì cập nhật và đẩy `(-p·w, v)`.

In [11]:
def dijkstra_max(n, edges, succProb, start_node, end_node):
    g = build_graph(n, edges, succProb)
    best = [0.0] * n
    best[start_node] = 1.0
    h = [(-1.0, start_node)]
    while h:
        negp, u = heapq.heappop(h)
        p = -negp
        if p < best[u]:
            continue                     # bản ghi lỗi thời
        if u == end_node:
            return p
        for v, w in g[u]:
            np_ = p * w
            if np_ > best[v]:
                best[v] = np_
                heapq.heappush(h, (-np_, v))
    return 0.0

for args, want in examples:
    print(round(dijkstra_max(*args), 5), "kỳ vọng", want)

0.25 kỳ vọng 0.25
0.3 kỳ vọng 0.3
0.0 kỳ vọng 0.0


In [12]:
class SolutionDijkstra:
    def maxProbability(self, n: int, edges: list[list[int]], succProb: list[float], start_node: int, end_node: int) -> float:
        g = [[] for _ in range(n)]
        for (a, b), p in zip(edges, succProb):
            g[a].append((b, p))
            g[b].append((a, p))
        best = [0.0] * n
        best[start_node] = 1.0
        h = [(-1.0, start_node)]
        while h:
            negp, u = heapq.heappop(h)
            p = -negp
            if u == end_node:
                return p
            if p < best[u]:
                continue
            for v, w in g[u]:
                if p * w > best[v]:
                    best[v] = p * w
                    heapq.heappush(h, (-best[v], v))
        return 0.0

print([round(SolutionDijkstra().maxProbability(*a), 5) for a, _ in examples])

[0.25, 0.3, 0.0]


**Độ phức tạp:** mỗi cạnh có thể đẩy một bản ghi → heap tối đa `O(E)` phần tử → Time `O((V + E) log E)`, Space `O(V + E)`.

### Cách 4 — Dijkstra trên trọng số `−log p` (theo gợi ý của đề)

Đề gợi ý: nhân nhiều xác suất có thể gây sai số; lấy log để cộng. Trọng số `w = −log p ≥ 0`; cạnh `p = 0` bị bỏ. Đáp án `exp(−dist[end])`.

In [13]:
def dijkstra_log(n, edges, succProb, start_node, end_node):
    g = [[] for _ in range(n)]
    for (a, b), p in zip(edges, succProb):
        if p > 0:
            w = -math.log(p)
            g[a].append((b, w))
            g[b].append((a, w))
    INF = float("inf")
    dist = [INF] * n
    dist[start_node] = 0.0
    h = [(0.0, start_node)]
    while h:
        d, u = heapq.heappop(h)
        if d > dist[u]:
            continue
        if u == end_node:
            return math.exp(-d)
        for v, w in g[u]:
            if d + w < dist[v]:
                dist[v] = d + w
                heapq.heappush(h, (dist[v], v))
    return 0.0

for args, want in examples:
    print(round(dijkstra_log(*args), 5), "kỳ vọng", want)

0.25 kỳ vọng 0.25
0.3 kỳ vọng 0.3
0.0 kỳ vọng 0.0


**Độ phức tạp:** như Cách 3, `O((V + E) log E)`; thêm chi phí `log/exp`. Ưu điểm: không bị **underflow** khi tích quá nhỏ (ví dụ `0.01¹⁰⁰⁰` = 0 trong float, nhưng tổng log vẫn biểu diễn được) — hữu ích khi cần so sánh các xác suất cực nhỏ.

In [14]:
tiny = 1.0
for _ in range(200):
    tiny *= 0.01
print("tích 0.01^200 bằng float:", tiny, "| tổng -log:", 200 * -math.log(0.01))

tích 0.01^200 bằng float: 0.0 | tổng -log: 921.0340371976182


## 8. TRACE từng bước

Trace Cách 3 (Dijkstra max-heap) trên đồ thị 5 đỉnh: cạnh `0–1:0.9`, `0–2:0.5`, `1–2:0.8`, `1–3:0.3`, `2–3:0.9`, `3–4:0.7`, đi từ 0 đến 4.

| pop (p, u) | bỏ qua? | cập nhật | best |
|---|---|---|---|
| (1.0, 0) | | 1←0.9, 2←0.5 | [1, .9, .5, 0, 0] |
| (0.9, 1) | | 2←0.72, 3←0.27 | [1, .9, .72, .27, 0] |
| (0.72, 2) | | 3←0.648 | [1, .9, .72, .648, 0] |
| (0.648, 3) | | 4←0.4536 | […, .4536] |
| (0.5, 2) | có (cũ) | | |
| (0.4536, 4) | → trả về | | |

In [15]:
def trace_dijkstra(n, edges, succProb, s, t):
    g = build_graph(n, edges, succProb)
    best = [0.0] * n
    best[s] = 1.0
    h = [(-1.0, s)]
    while h:
        negp, u = heapq.heappop(h)
        p = -negp
        if u == t:
            print(f"pop ({p:.4f}, {u}) = đích -> trả về {p:.4f}")
            return p
        if p < best[u]:
            print(f"pop ({p:.4f}, {u}) -> bỏ qua (best[{u}] = {best[u]:.4f})")
            continue
        ups = []
        for v, w in g[u]:
            if p * w > best[v]:
                best[v] = p * w
                heapq.heappush(h, (-best[v], v))
                ups.append(f"{v}<-{best[v]:.4f}")
        print(f"pop ({p:.4f}, {u}) cập nhật {ups} best={[round(b, 4) for b in best]}")
    return 0.0

E5 = [[0, 1], [0, 2], [1, 2], [1, 3], [2, 3], [3, 4]]
P5 = [0.9, 0.5, 0.8, 0.3, 0.9, 0.7]
trace_dijkstra(5, E5, P5, 0, 4)

pop (1.0000, 0) cập nhật ['1<-0.9000', '2<-0.5000'] best=[1.0, 0.9, 0.5, 0.0, 0.0]
pop (0.9000, 1) cập nhật ['2<-0.7200', '3<-0.2700'] best=[1.0, 0.9, 0.72, 0.27, 0.0]
pop (0.7200, 2) cập nhật ['3<-0.6480'] best=[1.0, 0.9, 0.72, 0.648, 0.0]
pop (0.6480, 3) cập nhật ['4<-0.4536'] best=[1.0, 0.9, 0.72, 0.648, 0.4536]
pop (0.5000, 2) -> bỏ qua (best[2] = 0.7200)
pop (0.4536, 4) = đích -> trả về 0.4536


0.45360000000000006

Trace Cách 2 (Bellman–Ford) trên cùng đồ thị — in `best` sau mỗi vòng:

In [16]:
def trace_bf(n, edges, succProb, s, t):
    best = [0.0] * n
    best[s] = 1.0
    for rnd in range(1, n):
        changed = False
        for (a, b), p in zip(edges, succProb):
            if best[a] * p > best[b]:
                best[b] = best[a] * p; changed = True
            if best[b] * p > best[a]:
                best[a] = best[b] * p; changed = True
        print(f"vòng {rnd}: best={[round(x, 4) for x in best]} changed={changed}")
        if not changed:
            break
    return best[t]

print("kết quả:", round(trace_bf(5, E5, P5, 0, 4), 4))

vòng 1: best=[1.0, 0.9, 0.72, 0.648, 0.4536] changed=True
vòng 2: best=[1.0, 0.9, 0.72, 0.648, 0.4536] changed=False
kết quả: 0.4536


## 9. Code nộp LeetCode

Bản chính: Dijkstra max-heap (Cách 3).

In [17]:
class Solution:
    def maxProbability(self, n: int, edges: list[list[int]], succProb: list[float], start_node: int, end_node: int) -> float:
        g = [[] for _ in range(n)]
        for (a, b), p in zip(edges, succProb):
            g[a].append((b, p))
            g[b].append((a, p))
        best = [0.0] * n
        best[start_node] = 1.0
        h = [(-1.0, start_node)]
        while h:
            negp, u = heapq.heappop(h)
            p = -negp
            if u == end_node:
                return p
            if p < best[u]:
                continue
            for v, w in g[u]:
                if p * w > best[v]:
                    best[v] = p * w
                    heapq.heappush(h, (-best[v], v))
        return 0.0

for args, want in examples:
    assert close(Solution().maxProbability(*args), want)
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

**Dijkstra (max):** Gọi `S` là tập đỉnh đã chốt. Bất biến: với mọi `u ∈ S`, `best[u]` là xác suất tối ưu; với `v ∉ S`, `best[v]` là xác suất tốt nhất trong các đường chỉ đi qua đỉnh của `S` rồi tới `v`.

Khi lấy ra `u ∉ S` có `best[u]` lớn nhất: giả sử có đường tốt hơn tới `u`. Đường đó rời `S` lần đầu tại đỉnh `x ∉ S`, nên xác suất của nó ≤ `best[x]·(tích phần còn lại)` ≤ `best[x]` ≤ `best[u]` (vì mọi xác suất ≤ 1 và `u` được chọn lớn nhất) — mâu thuẫn. Vậy `best[u]` tối ưu; khi `u = end` có thể trả về ngay.

**Bellman–Ford:** sau vòng thứ `k`, `best[v]` ≥ xác suất tốt nhất của mọi đường ≤ `k` cạnh tới `v`. Đường tối ưu là đường đơn (lặp đỉnh chỉ nhân thêm số ≤ 1) nên có ≤ `n − 1` cạnh. Mọi giá trị `best` luôn ứng với một đường thật → không vượt tối ưu. ∎

## 11. Phân tích Time / Space

`V = n ≤ 10⁴`, `E = len(edges) ≤ 2·10⁴`.

| Cách | Time | Space |
|---|---|---|
| 1. DFS mọi đường đơn | mũ, `O(n!)` worst | `O(n)` |
| 2. Bellman–Ford | worst `O(V·E)`; dừng sớm thường `O(k·E)` với `k` = số cạnh của đường tối ưu dài nhất | `O(V)` |
| 3. Dijkstra max-heap | `O((V + E) log E)` | `O(V + E)` |
| 4. Dijkstra `−log` | `O((V + E) log E)` | `O(V + E)` |

Dijkstra có thể dừng sớm khi pop được `end` — trung bình nhanh hơn nhiều nếu `end` gần `start`.

## 12. Bộ test

So sánh bằng sai số `1e-5`. Gồm: ví dụ đề, edge case (không có cạnh, cạnh xác suất 0, xác suất 1, nhiều đường bằng nhau), random so với brute DFS (seed cố định), test lớn `n = 10⁴`, `E = 2·10⁴`.

In [18]:
METHODS = {
    "bf": lambda a: SolutionBF().maxProbability(*a),
    "dijkstra": lambda a: Solution().maxProbability(*a),
    "dijkstra2": lambda a: SolutionDijkstra().maxProbability(*a),
    "log": lambda a: dijkstra_log(*a),
}
edge = [
    ((2, [], [], 0, 1), 0.0),
    ((2, [[0, 1]], [0.0], 0, 1), 0.0),
    ((2, [[1, 0]], [1.0], 0, 1), 1.0),
    ((4, [[0, 1], [1, 3], [0, 2], [2, 3]], [0.5, 0.4, 0.4, 0.5], 0, 3), 0.2),
    ((5, E5, P5, 0, 4), 0.9 * 0.8 * 0.9 * 0.7),
    ((5, E5, P5, 4, 0), 0.9 * 0.8 * 0.9 * 0.7),
]
for args, want in examples + edge:
    for name, f in METHODS.items():
        got = f(args)
        assert close(got, want), (name, args, got, want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [19]:
def random_graph(rng, n, m):
    pairs = [(a, b) for a in range(n) for b in range(a + 1, n)]
    rng.shuffle(pairs)
    edges = [list(p) if rng.random() < 0.5 else [p[1], p[0]] for p in pairs[:m]]
    probs = [rng.choice([0.0, 1.0, round(rng.random(), 3)]) if rng.random() < 0.15 else round(rng.random(), 3)
             for _ in edges]
    return edges, probs

rng = random.Random(1514)
for _ in range(400):
    n = rng.randint(2, 7)
    m = rng.randint(0, n * (n - 1) // 2)
    edges, probs = random_graph(rng, n, m)
    s, t = rng.sample(range(n), 2)
    args = (n, edges, probs, s, t)
    ref = brute(*args)
    for name, f in METHODS.items():
        assert close(f(args), ref), (name, args)
print("400 đồ thị ngẫu nhiên: mọi cách khớp brute (sai số 1e-5)")

400 đồ thị ngẫu nhiên: mọi cách khớp brute (sai số 1e-5)


In [20]:
rng = random.Random(99)
n = 10**4
edges, seen = [], set()
while len(edges) < 2 * 10**4:
    a, b = rng.randrange(n), rng.randrange(n)
    if a != b and (min(a, b), max(a, b)) not in seen:
        seen.add((min(a, b), max(a, b)))
        edges.append([a, b])
probs = [rng.uniform(0.5, 1.0) for _ in edges]
args = (n, edges, probs, 0, n - 1)
res = {}
for name in ["dijkstra", "log", "bf"]:
    t = time.perf_counter()
    res[name] = METHODS[name](args)
    print(f"{name:8s}: {res[name]:.6e} ({time.perf_counter() - t:.2f}s)")
assert close(res["dijkstra"], res["log"]) and close(res["dijkstra"], res["bf"])

dijkstra: 2.551530e-01 (0.02s)


log     : 2.551530e-01 (0.02s)
bf      : 2.551530e-01 (0.02s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên đồ thị **vô hướng** (chỉ thêm `a → b`)

In [21]:
def wrong_directed(n, edges, succProb, start_node, end_node):
    g = [[] for _ in range(n)]
    for (a, b), p in zip(edges, succProb):
        g[a].append((b, p))                          # SAI: thiếu g[b].append((a, p))
    best = [0.0] * n
    best[start_node] = 1.0
    h = [(-1.0, start_node)]
    while h:
        negp, u = heapq.heappop(h)
        if -negp < best[u]:
            continue
        for v, w in g[u]:
            if -negp * w > best[v]:
                best[v] = -negp * w
                heapq.heappush(h, (-best[v], v))
    return best[end_node]

args = (3, [[1, 0], [2, 1]], [0.5, 0.5], 0, 2)
print("sai:", wrong_directed(*args), "| đúng:", Solution().maxProbability(*args))

sai: 0.0 | đúng: 0.25


### Lỗi 2 — Dùng min-heap theo xác suất (quên đổi dấu) kèm "đã thăm"

Heap lấy đỉnh có xác suất **nhỏ nhất** trước → chốt sai.

In [22]:
def wrong_minheap(n, edges, succProb, start_node, end_node):
    g = build_graph(n, edges, succProb)
    best = [0.0] * n
    best[start_node] = 1.0
    done = [False] * n
    h = [(1.0, start_node)]
    while h:
        p, u = heapq.heappop(h)                       # SAI: lấy p NHỎ nhất
        if done[u]:
            continue
        done[u] = True
        for v, w in g[u]:
            if not done[v] and p * w > best[v]:
                best[v] = p * w
                heapq.heappush(h, (best[v], v))
    return best[end_node]

args = (5, E5, P5, 0, 4)
print("sai:", round(wrong_minheap(*args), 4), "| đúng:", round(Solution().maxProbability(*args), 4))

sai: 0.315 | đúng: 0.4536


### Lỗi 3 — Khởi tạo `best[start] = 0`

Mọi tích sau đó đều bằng 0 → luôn trả 0.

In [23]:
def wrong_init(n, edges, succProb, start_node, end_node):
    best = [0.0] * n                                   # SAI: quên best[start] = 1
    for _ in range(n - 1):
        for (a, b), p in zip(edges, succProb):
            best[b] = max(best[b], best[a] * p)
            best[a] = max(best[a], best[b] * p)
    return best[end_node]

for args, want in examples:
    print("sai:", wrong_init(*args), "| đúng:", want)

sai: 0.0 | đúng: 0.25
sai: 0.0 | đúng: 0.3
sai: 0.0 | đúng: 0.0


### Lỗi 4 — Cộng xác suất thay vì nhân

Xác suất đi qua nhiều cạnh độc lập là **tích**; cộng có thể ra số > 1.

In [24]:
def wrong_sum(n, edges, succProb, start_node, end_node):
    best = [-1.0] * n
    best[start_node] = 0.0
    for _ in range(n - 1):
        for (a, b), p in zip(edges, succProb):
            if best[a] >= 0:
                best[b] = max(best[b], best[a] + p)     # SAI: phải nhân
            if best[b] >= 0:
                best[a] = max(best[a], best[b] + p)
    return max(best[end_node], 0.0)

args = examples[0][0]
print("sai:", wrong_sum(*args), "| đúng:", Solution().maxProbability(*args))

sai: 2.6 | đúng: 0.25


### Lỗi 5 — So sánh kết quả số thực bằng `==` trong test

Đáp án đúng nhưng test báo sai vì sai số làm tròn.

In [25]:
got = Solution().maxProbability(3, [[0, 1], [1, 2]], [0.1, 0.3], 0, 2)
print("got =", got, "| got == 0.03 ?", got == 0.03, "| close ?", close(got, 0.03))

got = 0.03 | got == 0.03 ? True | close ? True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đường đi tốt nhất" với phép kết hợp đơn điệu** (cộng trọng số ≥ 0, nhân xác suất ≤ 1, lấy min/max của cạnh) → Dijkstra biến thể.
- **Tối đa hoá** → max-heap bằng cách đổi dấu.
- Có giới hạn số cạnh `K` → Bellman–Ford `K` vòng (LC 787).
- Đồ thị vô hướng → luôn thêm cạnh hai chiều.
- Kết quả số thực → test bằng sai số; đề có "accepted if differs by at most 1e-5".
- Có thể dừng Dijkstra ngay khi pop được đích.

## 15. Biến thể của bài

1. **LC 743 – Network Delay Time:** Dijkstra chuẩn (cộng trọng số).
2. **LC 1631 – Path With Minimum Effort:** kết hợp bằng `max` thay vì cộng.
3. **LC 787 – Cheapest Flights Within K Stops:** Bellman–Ford giới hạn số cạnh.
4. **LC 1976 – Number of Ways to Arrive at Destination:** Dijkstra + đếm số đường tối ưu.
5. Xác suất lớn nhất với tối đa `K` cạnh → Bellman–Ford `K` vòng, mỗi vòng dùng bản sao `best` cũ.
6. Truy vết đường đi → lưu `parent[v]` khi cập nhật.

In [26]:
# Minh hoạ biến thể 2: "minimax" — đường mà cạnh YẾU nhất mạnh nhất (kết hợp bằng min)
def max_bottleneck(n, edges, succProb, s, t):
    g = build_graph(n, edges, succProb)
    best = [0.0] * n
    best[s] = 1.0
    h = [(-1.0, s)]
    while h:
        negp, u = heapq.heappop(h)
        if -negp < best[u]:
            continue
        for v, w in g[u]:
            cand = min(-negp, w)
            if cand > best[v]:
                best[v] = cand
                heapq.heappush(h, (-cand, v))
    return best[t]

print("bottleneck 0->4:", max_bottleneck(5, E5, P5, 0, 4))

bottleneck 0->4: 0.7


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS mọi đường | Liệt kê đường đơn | mũ | `O(n)` | Đối chiếu, đồ thị rất nhỏ |
| 2. Bellman–Ford | Nới lỏng mọi cạnh tới khi ổn định | `O(V·E)` worst | `O(V)` | Dễ code, có giới hạn số cạnh |
| 3. Dijkstra max-heap | Chốt đỉnh có xác suất lớn nhất | `O(E log E)` | `O(V+E)` | Mặc định |
| 4. Dijkstra `−log` | Đưa về đường ngắn nhất chuẩn | `O(E log E)` | `O(V+E)` | Xác suất cực nhỏ, tái dùng code Dijkstra |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().maxProbability(2, [[0, 1]], [0.0], 0, 1)` và `Solution().maxProbability(3, [[0, 1], [1, 2], [0, 2]], [1.0, 1.0, 0.5], 0, 2)` trả về gì?

### Bài 2 (tính tay)
Đồ thị 4 đỉnh: `0–1: 0.6`, `1–3: 0.6`, `0–2: 0.9`, `2–3: 0.3`, `1–2: 0.9`. Tính xác suất lớn nhất từ 0 đến 3 và chỉ ra đường.

### Bài 3 (điền TODO)
Viết `best_path(n, edges, succProb, s, t)`: trả `(xác suất, danh sách đỉnh trên đường)`; nếu không có đường trả `(0.0, [])`.

### Bài 4 (tự viết lại)
Viết lại Bellman–Ford theo **danh sách kề** với hàng đợi (SPFA): `spfa(n, edges, succProb, s, t)`.

### Bài 5 (mini challenge)
Xác suất lớn nhất từ `s` đến `t` dùng **tối đa `K` cạnh**: `max_prob_k(n, edges, succProb, s, t, K)`.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def best_path(n, edges, succProb, s, t):
    # TODO: Dijkstra + parent[v] = u khi cập nhật; dựng ngược từ t
    pass

# Bài 4 — TODO
def spfa(n, edges, succProb, s, t):
    # TODO: deque các đỉnh vừa được cải thiện; in_queue để tránh trùng
    pass

# Bài 5 — TODO
def max_prob_k(n, edges, succProb, s, t, K):
    # TODO: K vòng Bellman-Ford, mỗi vòng đọc từ bản sao best của vòng trước
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Cạnh xác suất 0 không giúp gì. (2) Đường 0→1→2 có tích `1·1`. (3) `0.0` và `1.0`.

**Bài 2.** (1) Liệt kê các đường đơn 0→3. (2) Ứng viên: `0-1-3 = 0.36`, `0-2-1-3 = 0.9·0.9·0.6`. (3) `0.486` qua `0-2-1-3`.

**Bài 3.** (1) Mảng `parent = [-1]*n`. (2) Khi `best[v]` được cải thiện từ `u`, gán `parent[v] = u`. (3) Đi ngược từ `t` tới `s`, đảo danh sách.

**Bài 4.** (1) Bắt đầu với `deque([s])`. (2) Pop `u`, nới lỏng các cạnh kề; đỉnh nào được cải thiện mà chưa trong hàng đợi thì đẩy vào. (3) Trả `best[t]`.

**Bài 5.** (1) `best` ban đầu chỉ có `best[s] = 1`. (2) Mỗi vòng: `new = best[:]`, nới lỏng từ `best` (cũ) sang `new`. (3) Sau `K` vòng trả `best[t]`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2
r1 = Solution().maxProbability(2, [[0, 1]], [0.0], 0, 1)
r1b = Solution().maxProbability(3, [[0, 1], [1, 2], [0, 2]], [1.0, 1.0, 0.5], 0, 2)
E2 = [[0, 1], [1, 3], [0, 2], [2, 3], [1, 2]]
P2 = [0.6, 0.6, 0.9, 0.3, 0.9]
r2 = Solution().maxProbability(4, E2, P2, 0, 3)
print("Bài 1:", r1, r1b, "| Bài 2:", round(r2, 4))
assert r1 == 0.0 and close(r1b, 1.0) and close(r2, 0.486)

Bài 1: 0.0 1.0 | Bài 2: 0.486


In [29]:
# Bài 3
def best_path_sol(n, edges, succProb, s, t):
    g = build_graph(n, edges, succProb)
    best = [0.0] * n
    best[s] = 1.0
    parent = [-1] * n
    h = [(-1.0, s)]
    while h:
        negp, u = heapq.heappop(h)
        p = -negp
        if p < best[u]:
            continue
        if u == t:
            break
        for v, w in g[u]:
            if p * w > best[v]:
                best[v] = p * w
                parent[v] = u
                heapq.heappush(h, (-best[v], v))
    if best[t] == 0:
        return 0.0, []
    path = [t]
    while path[-1] != s:
        path.append(parent[path[-1]])
    return best[t], path[::-1]

def path_prob(edges, probs, path):
    w = {}
    for (a, b), p in zip(edges, probs):
        w[(a, b)] = w[(b, a)] = p
    return math.prod(w[(path[i], path[i + 1])] for i in range(len(path) - 1))

print(best_path_sol(4, E2, P2, 0, 3))
assert best_path_sol(4, E2, P2, 0, 3)[1] == [0, 2, 1, 3]

# Bài 4
def spfa_sol(n, edges, succProb, s, t):
    g = build_graph(n, edges, succProb)
    best = [0.0] * n
    best[s] = 1.0
    q = deque([s])
    inq = [False] * n
    inq[s] = True
    while q:
        u = q.popleft()
        inq[u] = False
        for v, w in g[u]:
            if best[u] * w > best[v]:
                best[v] = best[u] * w
                if not inq[v]:
                    inq[v] = True
                    q.append(v)
    return best[t]

rng = random.Random(3)
for _ in range(300):
    n = rng.randint(2, 7)
    edges, probs = random_graph(rng, n, rng.randint(0, n * (n - 1) // 2))
    s, t = rng.sample(range(n), 2)
    ref = brute(n, edges, probs, s, t)
    p, path = best_path_sol(n, edges, probs, s, t)
    assert close(p, ref)
    if path:
        assert path[0] == s and path[-1] == t and close(path_prob(edges, probs, path), p)
    assert close(spfa_sol(n, edges, probs, s, t), ref)
print("Bài 3, 4: OK")

(0.486, [0, 2, 1, 3])
Bài 3, 4: OK


In [30]:
# Bài 5
def max_prob_k_sol(n, edges, succProb, s, t, K):
    best = [0.0] * n
    best[s] = 1.0
    for _ in range(K):
        new = best[:]
        for (a, b), p in zip(edges, succProb):
            new[b] = max(new[b], best[a] * p)
            new[a] = max(new[a], best[b] * p)
        best = new
    return best[t]

def brute_k(n, edges, succProb, s, t, K):
    g = build_graph(n, edges, succProb)
    res = 0.0
    def dfs(u, prob, used, seen):
        nonlocal res
        if u == t:
            res = max(res, prob)
            return
        if used == K:
            return
        for v, w in g[u]:
            if v not in seen:
                dfs(v, prob * w, used + 1, seen | {v})
    dfs(s, 1.0, 0, {s})
    return res

print("K=1:", max_prob_k_sol(4, E2, P2, 0, 3, 1), "| K=2:", max_prob_k_sol(4, E2, P2, 0, 3, 2),
      "| K=3:", round(max_prob_k_sol(4, E2, P2, 0, 3, 3), 4))
for _ in range(300):
    n = rng.randint(2, 7)
    edges, probs = random_graph(rng, n, rng.randint(0, n * (n - 1) // 2))
    s, t = rng.sample(range(n), 2)
    K = rng.randint(1, n)
    assert close(max_prob_k_sol(n, edges, probs, s, t, K), brute_k(n, edges, probs, s, t, K))
print("Đáp án Bài 1–5: mọi assert đều qua")

K=1: 0.0 | K=2: 0.36 | K=3: 0.486
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template Dijkstra tối đa hoá tích (max-heap):**

```python
g = [[] for _ in range(n)]
for (a, b), p in zip(edges, probs):
    g[a].append((b, p)); g[b].append((a, p))      # vô hướng
best = [0.0] * n; best[s] = 1.0
h = [(-1.0, s)]
while h:
    negp, u = heapq.heappop(h); p = -negp
    if u == t: return p
    if p < best[u]: continue                      # bản ghi cũ
    for v, w in g[u]:
        if p * w > best[v]:
            best[v] = p * w
            heapq.heappush(h, (-best[v], v))
return 0.0
```

**Checklist trước khi submit:**
- [ ] Thêm cạnh **hai chiều**.
- [ ] `best[start] = 1.0`.
- [ ] Đẩy `-prob` vào heap (max-heap).
- [ ] Bỏ qua bản ghi lỗi thời (`p < best[u]`).
- [ ] Không có đường → trả `0.0`.
- [ ] Test số thực bằng sai số `1e-5`.
- [ ] Đúng chữ ký: `maxProbability(self, n, edges, succProb, start_node, end_node) -> float`.